In [ ]:
import os
from graphviz import Digraph
from IPython.display import Image, display


class Mini_Heap:
    def __init__(self, n):
        self.heap: list = [(nodeI, float("inf")) for nodeI in range(n)]
        self.nodeI2idx = {idx: idx for idx in range(n)}

    def __len__(self):
        """返回堆的大小 O(1)"""
        return len(self.heap)

    def __contains__(self, nodeI):
        """判断结点是否在堆中 O(1)"""
        return self.nodeI2idx[nodeI] != None

    def _update_nodeI2idx(self, idx):
        """更新结点的下标索引 O(1)"""
        self.nodeI2idx[self.heap[idx][0]] = idx

    def _heapify_down(self, idx):
        """向下调整最小堆 O(lgn)"""
        minI = idx
        size = len(self.heap)
        lchdI = 2 * idx + 1
        rchdI = 2 * idx + 2

        # 获取当前父子组最小结点
        if lchdI < size and self.heap[lchdI] < self.heap[minI]:
            minI = lchdI
        if rchdI < size and self.heap[rchdI] < self.heap[minI]:
            minI = rchdI

        # 如果最小结点不是根结点, 需要交换
        if minI != idx:
            self.heap[idx], self.heap[minI] = self.heap[minI], self.heap[idx]
            self._update_nodeI2idx(idx)
            self._update_nodeI2idx(minI)
            self._heapify_down(minI)

    def extract_min(self):
        """从堆中取出最小值结点 O(lgn)"""
        size = len(self.heap)

        # 将末尾结点交换到堆顶, 并更新其下标索引
        self.heap[0], self.heap[size - 1] = self.heap[size - 1], self.heap[0]
        self._update_nodeI2idx(0)

        # 弹出最小结点, 并标记为不在堆中
        min_nodeI = self.heap.pop()[0]
        self.nodeI2idx[min_nodeI] = None

        # 自顶向下调整堆 O(lgn)
        self._heapify_down(0)
        return min_nodeI

    def _heapify_up(self, idx):
        """向上调整最小堆 O(lgn)"""
        if idx == 0:
            return
        parI = (idx - 1) // 2
        if self.heap[parI] > self.heap[idx]:
            # 逐层上移更小结点, 并更新下标索引
            self.heap[parI], self.heap[idx] = self.heap[idx], self.heap[parI]
            self._update_nodeI2idx(parI)
            self._update_nodeI2idx(idx)
            self._heapify_up(parI)

    def get_key(self, nodeI):
        """获取堆中结点的对应值 O(1)"""
        idx = self.nodeI2idx[nodeI]
        if idx == None:
            print(f"Node {nodeI} not in heap")
        return self.heap[idx][1]

    def decrease_key(self, nodeI, new_key):
        """减小堆中结点的值 O(lgn)"""
        idx = self.nodeI2idx[nodeI]
        self.heap[idx] = (nodeI, new_key)
        self._heapify_up(idx)


class Node:
    def __init__(self, name):
        self.name = name
        self.par = None
        self.color = ""


def Dijk(adj):
    Q = Mini_Heap(len(adj))
    Q.decrease_key(0, 0)

    while len(Q) > 0:
        u = Q.extract_min()
        for v, w in adj[u]:
            d = Q.get_key(u) + w # TODO
            if v in Q and d < Q.get_key(v):
                Q.decrease_key(v, d)
                v.par = u
        draw(adj, Q)


def draw(adj, Q):
    dot = Digraph()
    dot.attr(rankdir="LR")
    dot.attr("node", shape="circle", fontsize="18")
    dot.attr("edge", fontsize="16")
    for u in adj:
        fontcolor = "white" if u.color == "black" else "black"
        u.d = Q.get_key(u)
        dot.node(u.name, f"{u.name}\n{"∞" if u.d == float('inf') else u.d}", style="filled", color=u.color, fontcolor=fontcolor)
        for v, cost in adj[u]:
            if v.par == u:
                dot.edge(u.name, v.name, label=str(cost), color="black", penwidth="4")
            else:
                dot.edge(u.name, v.name, label=str(cost))
    dot.render("demo", format="png", cleanup=True)
    display(Image("demo.png"))
    os.remove("demo.png")


# s, t, x, y, z = [Node(name) for name in "stxyz"]
# adj = {s: [(t, 10), (y, 5)], t: [(x, 1), (y, 2)], x: [(z, 4)], y: [(t, 3), (x, 9), (z, 2)], z: [(s, 7), (x, 6)]}
adj = [[(1, 10), (3, 5)], [(2, 1), (3, 2)], [(4, 4)], [(1, 3), (2, 9), (4, 2)], [(0, 7), (2, 6)]]
Dijk(adj)

Node 0 not in heap


TypeError: list indices must be integers or slices, not NoneType